In [1]:
from manim import *
import numpy as np

In [2]:
class Scene1PCAImportance(Scene):
    def construct(self):
        # --------------------------------------------------
        # 1. Load or define your data X here
        # --------------------------------------------------
        # X must have shape (N, d)
        # Example:
        # X = np.load("X.npy")
        
        np.random.seed(2)
        N, d = 250, 8
        X = np.random.randn(N, d)
        X[:, 1] = 0.8 * X[:, 0] + 0.3 * np.random.randn(N)
        X[:, 2] = 0.5 * X[:, 0] - 0.2 * X[:, 1] + 0.5 * np.random.randn(N)

        # --------------------------------------------------
        # 2. Center data and compute PCA
        # --------------------------------------------------
        Xc = X - X.mean(axis=0, keepdims=True)

        C = (Xc.T @ Xc) / Xc.shape[0]
        eigvals, eigvecs = np.linalg.eigh(C)

        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        # Scores on first two PCs
        Z = Xc @ eigvecs[:, :2]

        # Normalize for display
        Z = Z / np.max(np.abs(Z)) * 3.0

        # PCA feature importance using first k PCs
        k = min(3, d)
        pca_importance = np.sum(
            eigvals[:k][None, :] * eigvecs[:, :k] ** 2,
            axis=1
        )
        pca_importance = pca_importance / np.max(pca_importance)

        # --------------------------------------------------
        # 3. Title
        # --------------------------------------------------
        title = Text("Feature importance before training", font_size=36)
        subtitle = Text("PCA: data geometry only", font_size=26)
        subtitle.next_to(title, DOWN)

        self.play(Write(title), FadeIn(subtitle))
        self.wait(1)
        self.play(
            title.animate.to_edge(UP),
            subtitle.animate.next_to(title, DOWN)
        )

        # --------------------------------------------------
        # 4. Axes and point cloud
        # --------------------------------------------------
        axes = Axes(
            x_range=[-3.5, 3.5, 1],
            y_range=[-3.5, 3.5, 1],
            x_length=5.5,
            y_length=4.0,
            tips=False
        ).shift(LEFT * 3 + DOWN * 0.4)

        x_label = MathTex(r"\mathrm{PC}_1", font_size=28)
        y_label = MathTex(r"\mathrm{PC}_2", font_size=28)
        x_label.next_to(axes.x_axis, RIGHT)
        y_label.next_to(axes.y_axis, UP)

        points = VGroup()
        for z in Z:
            dot = Dot(
                axes.c2p(z[0], z[1]),
                radius=0.025
            )
            points.add(dot)

        cloud_label = Text("Data cloud projected on first PCs", font_size=22)
        cloud_label.next_to(axes, DOWN)

        self.play(Create(axes), FadeIn(x_label), FadeIn(y_label))
        self.play(LaggedStart(*[FadeIn(p) for p in points], lag_ratio=0.003))
        self.play(FadeIn(cloud_label))
        self.wait(1)

        # --------------------------------------------------
        # 5. Principal directions
        # --------------------------------------------------
        pc1_arrow = Arrow(
            axes.c2p(-2.7, 0),
            axes.c2p(2.7, 0),
            buff=0,
            stroke_width=6
        )
        pc2_arrow = Arrow(
            axes.c2p(0, -1.8),
            axes.c2p(0, 1.8),
            buff=0,
            stroke_width=4
        )

        pc1_text = MathTex(r"v_1", font_size=32).next_to(pc1_arrow, UP)
        pc2_text = MathTex(r"v_2", font_size=32).next_to(pc2_arrow, RIGHT)

        self.play(GrowArrow(pc1_arrow), Write(pc1_text))
        self.play(GrowArrow(pc2_arrow), Write(pc2_text))
        self.wait(1)

        # --------------------------------------------------
        # 6. Formula panel
        # --------------------------------------------------
        formula_box = RoundedRectangle(
            width=6.2,
            height=3.1,
            corner_radius=0.2
        ).shift(RIGHT * 3.2 + DOWN * 0.1)

        formula_title = Text("PCA feature importance", font_size=26)
        formula_title.move_to(formula_box.get_top() + DOWN * 0.45)

        formula = MathTex(
            r"I_j^{\mathrm{PCA}}",
            r"=",
            r"\sum_{\ell=1}^{k}",
            r"\lambda_\ell",
            r"v_{\ell j}^2",
            font_size=36
        )
        formula.move_to(formula_box.get_center() + UP * 0.25)

        explanation = VGroup(
            Text("depends only on", font_size=22),
            MathTex(r"X", font_size=34),
            Text("not on labels or training", font_size=22),
        ).arrange(RIGHT, buff=0.15)
        explanation.next_to(formula, DOWN, buff=0.45)

        self.play(Create(formula_box))
        self.play(Write(formula_title))
        self.play(Write(formula))
        self.play(FadeIn(explanation))
        self.wait(1)

        # --------------------------------------------------
        # 7. Feature importance bars
        # --------------------------------------------------
        bar_title = Text("Feature ranking from PCA", font_size=24)
        bar_title.move_to(RIGHT * 3.2 + DOWN * 2.05)

        bars = VGroup()
        labels = VGroup()

        max_bar_height = 1.1
        bar_width = 0.25
        spacing = 0.12

        start_x = 3.2 - (d * (bar_width + spacing)) / 2

        for j in range(d):
            height = max_bar_height * pca_importance[j]
            bar = Rectangle(
                width=bar_width,
                height=height
            )
            bar.move_to(
                np.array([
                    start_x + j * (bar_width + spacing),
                    -3.0 + height / 2,
                    0
                ])
            )

            label = MathTex(f"x_{{{j+1}}}", font_size=18)
            label.next_to(bar, DOWN, buff=0.1)

            bars.add(bar)
            labels.add(label)

        self.play(FadeIn(bar_title))
        self.play(LaggedStart(*[GrowFromEdge(bar, DOWN) for bar in bars], lag_ratio=0.08))
        self.play(FadeIn(labels))
        self.wait(1.5)

        # --------------------------------------------------
        # 8. Final takeaway
        # --------------------------------------------------
        takeaway = Text(
            "PCA asks: which features explain the geometry of the data?",
            font_size=28
        )
        takeaway.to_edge(DOWN)

        self.play(
            FadeOut(cloud_label),
            FadeIn(takeaway)
        )
        self.wait(2)

In [3]:
manim -pqh your_file.py Scene1PCAImportance

Manim Community v0.20.1

KeyError: ''